In [42]:
import os
import warnings
import numpy as np
import pandas as pd
import polars as pl
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb
from catboost import CatBoostClassifier
import xgboost as xgb

warnings.filterwarnings('ignore')

SEED = 42

def set_seed(seed=SEED):
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed()
def precision_at_recall(y_true: np.ndarray, y_scores: np.ndarray, target_recall: float = 0.70) -> float:
    y_true = np.asarray(y_true)
    y_scores = np.asarray(y_scores)

    desc_indices = np.argsort(y_scores)[::-1]
    y_true_sorted = y_true[desc_indices]
    y_scores_sorted = y_scores[desc_indices]

    distinct_value_indices = np.where(np.diff(y_scores_sorted))[0]
    threshold_indices = np.r_[distinct_value_indices, len(y_scores_sorted) - 1]

    tps = np.cumsum(y_true_sorted)[threshold_indices]
    fps = (threshold_indices + 1) - tps

    total_positives = np.sum(y_true)
    if total_positives == 0:
        return 0.0

    recalls = tps / total_positives
    precisions = tps / (tps + fps)

    valid_mask = recalls >= target_recall
    if not np.any(valid_mask):
        return 0.0

    return float(np.max(precisions[valid_mask]))


print("1/6. Загрузка метаданных...")
train_df = pd.read_csv('data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test_df = pd.read_csv('data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])

events_lazy = pl.scan_csv('data/events.csv.gz', try_parse_dates=True)

meta_all = pd.concat([
    train_df[['cookie_id', 'window_start_ts', 'window_end_ts']],
    test_df[['cookie_id', 'window_start_ts', 'window_end_ts']]
], ignore_index=True)

meta_pl = pl.from_pandas(meta_all)

print("2/6. Генерация расширенного набора признаков (Polars)...")

BOT_KEYWORDS = "(?i)(bot|crawler|spider|headless|selenium|puppeteer|phantom|python|curl|wget|requests|axios|scrapy)"

events_filtered = (
    events_lazy
    .join(meta_pl.lazy(), on="cookie_id", how="inner")
    .filter((pl.col("event_ts") >= pl.col("window_start_ts")) & (pl.col("event_ts") < pl.col("window_end_ts")))
    .sort(["cookie_id", "event_ts"])
    .with_columns([
        (pl.col("event_ts").diff().over("cookie_id").dt.total_milliseconds() / 1000.0).alias("dt"),

        (((pl.col("pointer_x").diff().over("cookie_id")) ** 2 +
          (pl.col("pointer_y").diff().over("cookie_id")) ** 2) ** 0.5).alias("pointer_dist")
    ])
    .with_columns([
        (pl.col("pointer_dist") / (pl.col("dt") + 0.001)).alias("pointer_speed")
    ])
    .with_columns([
        (pl.col("pointer_speed").diff().over("cookie_id").abs()).alias("pointer_accel")
    ])
)

features_base = (
    events_filtered
    .group_by("cookie_id")
    .agg([
        pl.len().alias("n_events"),
        pl.col("item_id").n_unique().alias("item_nunique"),
        pl.col("item_category").n_unique().alias("category_nunique"),
        pl.col("item_location").n_unique().alias("location_nunique"),
        pl.col("seller_type").n_unique().alias("seller_type_nunique"),
        pl.col("search_query").n_unique().alias("query_nunique"),
        pl.col("user_agent").n_unique().alias("ua_nunique"),
        pl.col("platform").n_unique().alias("platform_nunique"),

        pl.col("user_agent").first().alias("top_user_agent"),
        pl.col("platform").first().alias("top_platform"),

        pl.col("dt").mean().alias("dt_mean"),
        pl.col("dt").std().alias("dt_std"),
        pl.col("dt").median().alias("dt_median"),
        pl.col("dt").quantile(0.05).alias("dt_q05"),
        pl.col("dt").quantile(0.10).alias("dt_q10"),
        pl.col("dt").quantile(0.25).alias("dt_q25"),
        pl.col("dt").quantile(0.75).alias("dt_q75"),
        pl.col("dt").quantile(0.90).alias("dt_q90"),
        pl.col("dt").quantile(0.95).alias("dt_q95"),
        pl.col("dt").max().alias("dt_max"),

        (pl.col("dt") == 0).mean().alias("dt_zero_ratio"),
        (pl.col("dt") < 0.02).mean().alias("dt_ultra_fast_20ms_ratio"),
        (pl.col("dt") < 0.10).mean().alias("dt_fast_100ms_ratio"),
        (pl.col("dt") > 10.0).mean().alias("dt_pause_ratio"),

        pl.col("search_page").max().alias("search_page_max"),
        pl.col("search_page").mean().alias("search_page_mean"),
        pl.col("search_page").n_unique().alias("search_page_nunique"),
        (pl.col("search_page") > 3).mean().alias("deep_search_ratio"),

        pl.col("pointer_dist").mean().alias("pointer_dist_mean"),
        pl.col("pointer_dist").max().alias("pointer_dist_max"),
        pl.col("pointer_dist").std().alias("pointer_dist_std"),
        pl.col("pointer_speed").mean().alias("pointer_speed_mean"),
        pl.col("pointer_speed").max().alias("pointer_speed_max"),
        pl.col("pointer_accel").mean().alias("pointer_accel_mean"),
        (pl.col("pointer_dist") == 0).mean().alias("pointer_static_ratio"),
        (pl.col("pointer_x").is_null()).mean().alias("pointer_null_ratio"),

        ((pl.col("pointer_x").max() - pl.col("pointer_x").min()) *
         (pl.col("pointer_y").max() - pl.col("pointer_y").min())).alias("pointer_bbox_area"),

        pl.col("user_agent").str.contains(BOT_KEYWORDS).max().cast(pl.Int32).alias("ua_has_bot_keyword"),
        pl.col("user_agent").first().str.len_bytes().alias("ua_length")
    ])
)

events_pivoted = (
    events_filtered
    .group_by(["cookie_id", "event_name"])
    .len()
    .collect()
    .pivot(on="event_name", index="cookie_id", values="len")
    .fill_null(0)
)
event_cols = [c for c in events_pivoted.columns if c != "cookie_id"]
events_pivoted = events_pivoted.rename({c: f"ev_cnt_{c}" for c in event_cols})

features_pl = features_base.collect().join(events_pivoted, on="cookie_id", how="left").fill_null(0)
features_df = features_pl.to_pandas()


print("3/6. Формирование финальных признаков и агрегаций...")
train_full = train_df.merge(features_df, on="cookie_id", how="left").fillna(0)
test_full = test_df.merge(features_df, on="cookie_id", how="left").fillna(0)

full_df = pd.concat([train_full, test_full], ignore_index=True)

ua_counts = full_df['top_user_agent'].value_counts().to_dict()
platform_counts = full_df['top_platform'].value_counts().to_dict()

for df in [train_full, test_full]:
    df['top_user_agent'] = df['top_user_agent'].fillna('unknown').astype(str)
    df['top_platform'] = df['top_platform'].fillna('unknown').astype(str)

    df['cookie_age'] = (df['window_end_ts'] - df['cookie_created_at']).dt.total_seconds()
    df['created_to_window_start'] = (df['window_start_ts'] - df['cookie_created_at']).dt.total_seconds()
    df['window_duration'] = (df['window_end_ts'] - df['window_start_ts']).dt.total_seconds()
    df['is_fresh_cookie'] = (df['created_to_window_start'] < 300).astype(int)

    df['events_per_sec'] = df['n_events'] / (df['window_duration'] + 1e-3)
    df['dt_cv'] = df['dt_std'] / (df['dt_mean'] + 1e-5)
    df['events_per_item'] = df['n_events'] / (df['item_nunique'] + 1)
    df['items_per_event'] = df['item_nunique'] / (df['n_events'] + 1)
    df['categories_per_item'] = df['category_nunique'] / (df['item_nunique'] + 1)

    df['ua_freq'] = df['top_user_agent'].map(ua_counts).fillna(0)
    df['platform_freq'] = df['top_platform'].map(platform_counts).fillna(0)

    ev_count_columns = [c for c in df.columns if c.startswith('ev_cnt_')]
    for col in ev_count_columns:
        pct_name = col.replace('ev_cnt_', 'ev_pct_')
        df[pct_name] = df[col] / (df['n_events'] + 1e-5)

exclude_cols = ['cookie_id', 'target', 'cookie_created_at', 'window_start_ts', 'window_end_ts', 'top_platform']
feature_cols = [c for c in train_full.columns if c not in exclude_cols]

print(f"Сформировано признаков: {len(feature_cols)}")

print("4/6. Обучение ансамбля (LightGBM + CatBoost + XGBoost) на 5-Fold Stratified CV...")

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

X = train_full[feature_cols].copy()
y = train_full['target'].values
X_test = test_full[feature_cols].copy()

X_lgb_xgb = X.drop(columns=['top_user_agent'])
X_test_lgb_xgb = X_test.drop(columns=['top_user_agent'])

oof_lgb = np.zeros(len(train_full))
oof_cb = np.zeros(len(train_full))
oof_xgb = np.zeros(len(train_full))

test_preds_lgb = np.zeros(len(test_full))
test_preds_cb = np.zeros(len(test_full))
test_preds_xgb = np.zeros(len(test_full))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr_num, y_tr = X_lgb_xgb.iloc[train_idx], y[train_idx]
    X_va_num, y_va = X_lgb_xgb.iloc[val_idx], y[val_idx]

    X_tr_cb, X_va_cb = X.iloc[train_idx], X.iloc[val_idx]

    model_lgb = lgb.LGBMClassifier(
        n_estimators=2000,
        learning_rate=0.015,
        num_leaves=63,
        max_depth=7,
        subsample=0.8,
        colsample_bytree=0.7,
        min_child_samples=30,
        random_state=SEED + fold,
        n_jobs=-1,
        verbose=-1
    )
    model_lgb.fit(
        X_tr_num, y_tr,
        eval_set=[(X_va_num, y_va)],
        callbacks=[lgb.early_stopping(100, verbose=False)]
    )
    oof_lgb[val_idx] = model_lgb.predict_proba(X_va_num)[:, 1]
    test_preds_lgb += model_lgb.predict_proba(X_test_lgb_xgb)[:, 1] / N_SPLITS

    model_cb = CatBoostClassifier(
        iterations=1800,
        learning_rate=0.02,
        depth=6,
        l2_leaf_reg=5.0,
        text_features=['top_user_agent'],
        random_seed=SEED + fold,
        thread_count=-1,
        verbose=0
    )
    model_cb.fit(
        X_tr_cb, y_tr,
        eval_set=(X_va_cb, y_va),
        early_stopping_rounds=100
    )
    oof_cb[val_idx] = model_cb.predict_proba(X_va_cb)[:, 1]
    test_preds_cb += model_cb.predict_proba(X_test)[:, 1] / N_SPLITS

    # --- 3. XGBoost ---
    model_xgb = xgb.XGBClassifier(
        n_estimators=1500,
        learning_rate=0.02,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.7,
        random_state=SEED + fold,
        n_jobs=-1,
        eval_metric='logloss',
        early_stopping_rounds=100
    )
    model_xgb.fit(
        X_tr_num, y_tr,
        eval_set=[(X_va_num, y_va)],
        verbose=False
    )
    oof_xgb[val_idx] = model_xgb.predict_proba(X_va_num)[:, 1]
    test_preds_xgb += model_xgb.predict_proba(X_test_lgb_xgb)[:, 1] / N_SPLITS

score_lgb_oof = precision_at_recall(y, oof_lgb)
score_cb_oof = precision_at_recall(y, oof_cb)
score_xgb_oof = precision_at_recall(y, oof_xgb)

print(f"\n--- Результаты на OOF (Out-Of-Fold) ---")
print(f"LGBM P@R>=0.70:     {score_lgb_oof:.4f}")
print(f"CatBoost P@R>=0.70: {score_cb_oof:.4f}")
print(f"XGBoost P@R>=0.70:  {score_xgb_oof:.4f}")
print("\n5/6. Оптимизация весов блендинга...")

best_score = -1.0
best_weights = (0.33, 0.33, 0.34)

for w1 in np.linspace(0, 1, 101):
    for w2 in np.linspace(0, 1 - w1, 101):
        w3 = 1.0 - w1 - w2

        if w3 < -1e-5:
            continue

        blend = w1 * oof_lgb + w2 * oof_cb + w3 * oof_xgb
        score = precision_at_recall(y, blend)

        if score > best_score:
            best_score = score
            best_weights = (w1, w2, w3)

w1, w2, w3 = best_weights
oof_blend = w1 * oof_lgb + w2 * oof_cb + w3 * oof_xgb

print(f"Оптимальные веса: LGBM={w1:.3f}, CatBoost={w2:.3f}, XGBoost={w3:.3f}")
print(f"==========================================")
print(f">>> Ансамбль OOF P@R>=0.70: {best_score:.4f} <<<")
print(f"==========================================\n")

print("6/6. Формирование итогового файла submission.csv...")

final_test_scores = w1 * test_preds_lgb + w2 * test_preds_cb + w3 * test_preds_xgb

submission = pd.DataFrame({
    'cookie_id': test_full['cookie_id'],
    'score': final_test_scores
})

submission.to_csv('submission.csv', index=False)
print("Файл submission.csv успешно сохранен!")

assert len(submission) == len(test_df), "Несовпадение числа строк в test.csv!"
assert submission['score'].isnull().sum() == 0, "Есть NaN в значениях score!"
print(f"Итоговая проверка прошла успешно! Строк: {len(submission)}")



1/6. Загрузка метаданных...
2/6. Генерация расширенного набора признаков (Polars)...
3/6. Формирование финальных признаков и агрегаций...
Сформировано признаков: 67
4/6. Обучение ансамбля (LightGBM + CatBoost + XGBoost) на 5-Fold Stratified CV...

--- Результаты на OOF (Out-Of-Fold) ---
LGBM P@R>=0.70:     0.7066
CatBoost P@R>=0.70: 0.7192
XGBoost P@R>=0.70:  0.7233

5/6. Оптимизация весов блендинга...
Оптимальные веса: LGBM=0.000, CatBoost=0.630, XGBoost=0.370
>>> Ансамбль OOF P@R>=0.70: 0.7485 <<<

6/6. Формирование итогового файла submission.csv...
Файл submission.csv успешно сохранен!
Итоговая проверка прошла успешно! Строк: 4909
